# Dunnhumby Complete Journey — Fixed Preprocessing Pipeline
# Feeds CrossDemandModel for the F2T Dynamic Pricing QMIX agent.
#
# Fixes applied vs original notebook:
#   1. Exclude non-produce: POPCORN, NUTS, PROCESSED, SEASONAL, DRIED FRUIT
#   2. Revenue-based demand (solves unit mixing / potato problem)
#   3. Product fixed-effects elasticity (solves endogeneity)
#   4. Non-promoted filter + week traffic control
#   5. Literature overrides for broken positive-beta estimates
#   6. Fourier seasonality fitted per sub-category
#   7. Fixed cross-price bug: was 1.0**E[i,j], now uses actual prices
#   8. Fixed-effects cross-elasticity matrix
#   9. Consolidation to 4-category FINAL_PARAMS
#  10. Save step producing demand_params.json + cross_elasticity_matrix.npy


In [1]:
import pandas as pd
import numpy as np
import json, os, warnings
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')

trans = pd.read_csv('transaction_data.csv')
prod  = pd.read_csv('product.csv')
caus  = pd.read_csv('causal_data.csv')

print("Transaction columns:", trans.columns.tolist())
print("Product columns:    ", prod.columns.tolist())
print("Causal columns:     ", caus.columns.tolist())
print(f"\nTotal transactions: {len(trans):,}")
print(f"Total products:     {len(prod):,}")

Transaction columns: ['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']
Product columns:     ['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']
Causal columns:      ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']

Total transactions: 2,595,732
Total products:     92,353


In [2]:
# Inspect BEFORE mapping so you know what's actually in the data
produce_raw = prod[prod['DEPARTMENT'] == 'PRODUCE'].copy()
print(f"Products in PRODUCE dept: {len(produce_raw)}")
print("\nCOMMODITY_DESC value counts:")
print(produce_raw['COMMODITY_DESC'].value_counts().to_string())

Products in PRODUCE dept: 3118

COMMODITY_DESC value counts:
COMMODITY_DESC
PROCESSED                       319
ORGANICS FRUIT & VEGETABLES     308
SALAD MIX                       253
VEGETABLES - ALL OTHERS         203
POPCORN                         199
APPLES                          173
CITRUS                          154
POTATOES                        142
ONIONS                          117
NUTS                            112
BERRIES                         105
VALUE ADDED VEGETABLES           96
HERBS                            91
TOMATOES                         85
TROPICAL FRUIT                   81
MUSHROOMS                        80
VALUE ADDED FRUIT                79
PEPPERS-ALL                      66
STONE FRUIT                      60
VEGETABLES SALAD                 60
CARROTS                          57
GRAPES                           51
MELONS                           49
BROCCOLI/CAULIFLOWER             48
SQUASH                           36
PEARS                   

In [3]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — Exclusions + category mapping
# ─────────────────────────────────────────────────────────────────

# FIX 1: Explicitly exclude non-fresh-produce commodities
EXCLUDE_COMMODITIES = {
    'POPCORN',                    # 199 products — snack food, not produce
    'NUTS',                       # 112 products — shelf-stable, not fresh
    'DRIED FRUIT',                # not fresh
    'PROCESSED',                  # jarred, pickled, shelf-stable
    'SEASONAL',                   # non-food seasonal items
    'PROD SUPPLIES',              # bags, twist ties, etc.
    'SALAD BAR',                  # prepared food, not raw produce
    'MISCELLANEOUS(CORP USE ONLY)',
}

COMMODITY_MAP = {
    # ── Leafy greens ────────────────────────────────────────────
    'SALAD MIX':                 'leafy',
    'VEGETABLES - ALL OTHERS':   'leafy',
    'VALUE ADDED VEGETABLES':    'leafy',
    'VEGETABLES SALAD':          'leafy',
    'BROCCOLI/CAULIFLOWER':      'leafy',
    'PEPPERS-ALL':               'leafy',
    'CORN':                      'leafy',
    'MUSHROOMS':                 'leafy',  # close enough — no separate category
    # ── Root vegetables ─────────────────────────────────────────
    'POTATOES':                  'root',
    'ONIONS':                    'root',
    'CARROTS':                   'root',
    'SQUASH':                    'root',
    # ── Fruit ───────────────────────────────────────────────────
    'APPLES':                    'fruit',
    'CITRUS':                    'fruit',
    'BERRIES':                   'fruit',
    'TOMATOES':                  'fruit',
    'TROPICAL FRUIT':            'fruit',
    'VALUE ADDED FRUIT':         'fruit',
    'STONE FRUIT':               'fruit',
    'GRAPES':                    'fruit',
    'MELONS':                    'fruit',
    'PEARS':                     'fruit',
    # ── Herbs ───────────────────────────────────────────────────
    'HERBS':                     'herbs',
}

def map_organics(sub_commodity):
    """ORGANICS FRUIT & VEGETABLES — determine category from sub-commodity."""
    s = str(sub_commodity).upper()
    if any(x in s for x in ['APPLE','BERRY','BERRIES','GRAPE','CITRUS',
                             'FRUIT','MANGO','MELON','PEACH','CHERRY',
                             'STRAWBERR','BLUEBERR','RASPBERR','TOMATO']):
        return 'fruit'
    if any(x in s for x in ['HERB','BASIL','CILANT','PARSLEY','DILL']):
        return 'herbs'
    if any(x in s for x in ['CARROT','POTATO','ONION','MUSHROOM','SQUASH','BEET']):
        return 'root'
    return 'leafy'  # default for organic veg

# Apply exclusions
produce_products = produce_raw[
    ~produce_raw['COMMODITY_DESC'].isin(EXCLUDE_COMMODITIES)
].copy()

# Apply commodity-level mapping
produce_products['category'] = produce_products['COMMODITY_DESC'].map(COMMODITY_MAP)

# Handle ORGANICS FRUIT & VEGETABLES separately
mask = produce_products['COMMODITY_DESC'] == 'ORGANICS FRUIT & VEGETABLES'
produce_products.loc[mask, 'category'] = (
    produce_products.loc[mask, 'SUB_COMMODITY_DESC'].apply(map_organics)
)

# Report unmapped (review these)
unmapped = produce_products[produce_products['category'].isna()]['COMMODITY_DESC'].value_counts()
if len(unmapped):
    print("Unmapped commodities — add to COMMODITY_MAP or EXCLUDE_COMMODITIES:")
    print(unmapped.to_string())

produce_products = produce_products.dropna(subset=['category'])
print(f"Produce products after filtering: {len(produce_products)}")
print("\nCategory distribution:")
print(produce_products['category'].value_counts())

Produce products after filtering: 2453

Category distribution:
category
fruit    964
leafy    961
root     405
herbs    123
Name: count, dtype: int64


In [4]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Parse and standardise product size
# Solves the unit mixing problem (bulk vs bagged)
# ─────────────────────────────────────────────────────────────────
import re

def parse_product_size(size_str):
    if pd.isna(size_str) or str(size_str).strip() == '':
        return 1.0, 'UNKNOWN'
    s = str(size_str).upper().strip()
    m = re.search(r'([0-9\.]+)\s*([A-Z]+)', s)
    if m:
        try:
            return float(m.group(1)), m.group(2)
        except ValueError:
            return 1.0, 'UNKNOWN'
    return 1.0, 'EACH'

produce_products[['size_value','size_unit']] = (
    produce_products['CURR_SIZE_OF_PRODUCT']
    .apply(lambda x: pd.Series(parse_product_size(x)))
)

def normalize_to_lbs(row):
    if row['size_unit'] in ('OZ', 'OUNCE'):
        return row['size_value'] / 16.0, 'LB'
    if row['size_unit'] in ('LBS','LB'):
        return row['size_value'], 'LB'
    return row['size_value'], row['size_unit']

produce_products[['normalized_size','normalized_unit']] = produce_products.apply(
    lambda r: pd.Series(normalize_to_lbs(r)), axis=1
)

def clean_final_units(u):
    u = str(u).upper()
    if u in ('CT','EACH','PK','PACK','CTN','BU','BUNCHES','ML'): return 'UNIT'
    if u in ('PT','PTS'): return 'PINT'
    if u == 'UNKNOWN': return 'BULK_OR_UNKNOWN'
    if u == 'OUNCE': return 'LB'
    return u

produce_products['final_unit'] = produce_products['normalized_unit'].apply(clean_final_units)
print("Final unit distribution:")
print(produce_products['final_unit'].value_counts())

Final unit distribution:
final_unit
LB                 1452
BULK_OR_UNKNOWN     496
UNIT                471
PINT                 34
Name: count, dtype: int64


In [5]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — Join transactions to produce products
# ─────────────────────────────────────────────────────────────────
trans_produce = trans.merge(
    produce_products[['PRODUCT_ID','COMMODITY_DESC','category',
                       'SUB_COMMODITY_DESC','normalized_size','final_unit']],
    on='PRODUCT_ID', how='inner'
)
print(f"Total transactions:   {len(trans):,}")
print(f"Produce transactions: {len(trans_produce):,}")
print(f"Produce share:        {100*len(trans_produce)/len(trans):.1f}%")
print("\nTransactions by category:")
print(trans_produce['category'].value_counts())

Total transactions:   2,595,732
Produce transactions: 245,175
Produce share:        9.4%

Transactions by category:
category
fruit    118620
leafy     76832
root      45254
herbs      4469
Name: count, dtype: int64


In [6]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — Compute shelf price (official Dunnhumby formula)
# shelf_price = (SALES_VALUE + all_discounts) / QUANTITY
# This is what was on the price tag before any discount applied.
# Using SALES_VALUE/QUANTITY alone gives loyalty card price — wrong
# for elasticity fitting (confounds price with discount response).
# ─────────────────────────────────────────────────────────────────
trans_produce = trans_produce[trans_produce['QUANTITY'] > 0].copy()

total_disc = (
    trans_produce['RETAIL_DISC'].abs() +
    trans_produce['COUPON_DISC'].abs() +
    trans_produce['COUPON_MATCH_DISC'].abs()
)
trans_produce['base_price'] = (
    (trans_produce['SALES_VALUE'] + total_disc) / trans_produce['QUANTITY']
)

# Standard price: normalise LB items to price-per-LB
trans_produce['standard_price'] = np.where(
    trans_produce['final_unit'] == 'LB',
    trans_produce['base_price'] / trans_produce['normalized_size'].clip(lower=0.01),
    trans_produce['base_price']
)

# IQR outlier filter per category × unit (keeps price spread realistic)
def filter_iqr(df, col):
    Q1  = df.groupby(['category','final_unit'])[col].transform('quantile', 0.25)
    Q3  = df.groupby(['category','final_unit'])[col].transform('quantile', 0.75)
    IQR = Q3 - Q1
    return df[(df[col] >= 0.05) & (df[col] <= Q3 + 1.5*IQR)].copy()

trans_produce = filter_iqr(trans_produce, 'standard_price')

# Herbs LB price floor: remove outlier micro-prices after IQR filter
herbs_lb = (trans_produce['category']=='herbs') & (trans_produce['final_unit']=='LB')
trans_produce = trans_produce[~(herbs_lb & (trans_produce['standard_price'] < 0.50))]

print("Standard price distribution by category × unit:")
print(trans_produce.groupby(['category','final_unit'])['standard_price']
      .describe().round(2).to_string())

Standard price distribution by category × unit:
                            count  mean   std   min   25%   50%    75%    max
category final_unit                                                          
fruit    BULK_OR_UNKNOWN  34358.0  2.01  1.32  0.05  0.87  1.79   2.92   6.14
         LB               29768.0  0.42  0.42  0.05  0.15  0.23   0.62   2.14
         PINT               686.0  2.85  0.32  1.00  2.99  2.99   2.99   2.99
         UNIT              9285.0  2.61  1.41  0.20  1.69  2.50   3.49   6.05
herbs    BULK_OR_UNKNOWN     85.0  1.38  0.25  0.25  1.49  1.49   1.49   1.49
         LB                1493.0  9.15  3.62  0.51  5.30  9.96  10.61  19.14
         UNIT              1143.0  1.25  0.26  0.39  0.99  1.49   1.49   1.49
leafy    BULK_OR_UNKNOWN  11503.0  1.63  0.43  0.13  1.29  1.69   1.99   3.04
         LB               28871.0  3.45  1.77  0.05  1.99  3.98   4.78   8.91
         PINT                24.0  2.49  0.00  2.49  2.49  2.49   2.49   2.49
         UNIT   

In [7]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — Promotion flags from causal_data
# ─────────────────────────────────────────────────────────────────
caus['display']  = caus['display'].astype(str)
caus['mailer']   = caus['mailer'].astype(str)
caus['promoted'] = ((caus['display'] != '0') | (caus['mailer'] != '0')).astype(int)

caus_flag = caus[['PRODUCT_ID','STORE_ID','WEEK_NO','promoted']].drop_duplicates(
    subset=['PRODUCT_ID','STORE_ID','WEEK_NO']
)

if 'promoted' in trans_produce.columns:
    trans_produce = trans_produce.drop(columns=['promoted'])

trans_produce = trans_produce.merge(
    caus_flag, on=['PRODUCT_ID','STORE_ID','WEEK_NO'], how='left'
)
trans_produce['promoted'] = trans_produce['promoted'].fillna(0).astype(int)

print("Promotion rate by category:")
print(trans_produce.groupby('category')['promoted'].mean().round(3))

Promotion rate by category:
category
fruit    0.182
herbs    0.016
leafy    0.139
root     0.098
Name: promoted, dtype: float64


In [8]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — Time features
# DAY=1 → Thursday (confirmed from Dunnhumby user guide)
# TRANS_TIME → HHMM integer (1423 = 14:23)
# ─────────────────────────────────────────────────────────────────
trans_produce['hour_of_day'] = (trans_produce['TRANS_TIME'] // 100).astype('int8')
assert trans_produce['hour_of_day'].between(0, 23).all(), "TRANS_TIME parse error"

# 0=Thu, 1=Fri, 2=Sat, 3=Sun, 4=Mon, 5=Tue, 6=Wed
trans_produce['day_of_week'] = ((trans_produce['DAY'] - 1) % 7).astype('int8')
trans_produce['hour_of_week'] = (
    trans_produce['day_of_week'] * 24 + trans_produce['hour_of_day']
).astype('int16')

# Time segment for inspection (not used in modelling)
bins       = [-1, 4, 11, 17, 20, 23]
labels_ts  = ['Night','Morning','Afternoon','Evening','Night2']
trans_produce['time_segment'] = pd.cut(
    trans_produce['hour_of_day'], bins=bins, labels=labels_ts
).astype(str).replace({'Night2': 'Night'})

print("Transactions by time segment (%):")
print((trans_produce['time_segment'].value_counts(normalize=True)*100).round(1))
print("\nNOTE: US data is Afternoon-heavy. VN is Morning-heavy.")
print("Fourier phase will be adjusted during domain randomization.")

Transactions by time segment (%):
time_segment
Afternoon    54.2
Evening      22.8
Morning      15.4
Night         7.5
Name: proportion, dtype: float64

NOTE: US data is Afternoon-heavy. VN is Morning-heavy.
Fourier phase will be adjusted during domain randomization.


In [9]:
# ─────────────────────────────────────────────────────────────────
# STEP 8 — Aggregate to hourly demand
# FIX: Use REVENUE (SALES_VALUE) not QUANTITY as demand signal.
# Reason: QUANTITY mixes units — 1 bag potatoes vs 5 lbs bulk
# potatoes both represent the same purchase but show quantity=1
# vs quantity=5. Revenue is unit-agnostic.
# ─────────────────────────────────────────────────────────────────
hourly = (
    trans_produce
    .groupby(['WEEK_NO','hour_of_week','category','final_unit'])
    .agg(
        revenue    = ('SALES_VALUE',    'sum'),
        n_baskets  = ('BASKET_ID',     'nunique'),
        avg_price  = ('standard_price', 'mean'),
        promo_rate = ('promoted',       'mean'),
    )
    .reset_index()
)

# Demand rate: revenue per basket session (unit-agnostic)
hourly['demand_rate'] = hourly['revenue'] / hourly['n_baskets'].clip(lower=1)

# Remove very sparse hours (fewer than 3 baskets — noise)
hourly = hourly[hourly['n_baskets'] >= 3].copy()

# Log-normalise relative to category × unit mean
hourly['ref_price']  = hourly.groupby(['category','final_unit'])['avg_price'].transform('mean')
hourly['ref_demand'] = hourly.groupby(['category','final_unit'])['demand_rate'].transform('mean')
hourly['log_p_norm'] = np.log(hourly['avg_price']   / hourly['ref_price']  + 1e-9)
hourly['log_d_norm'] = np.log(hourly['demand_rate'] / hourly['ref_demand'] + 1e-9)

print("Demand rate distribution by category × unit:")
print(hourly.groupby(['category','final_unit'])['demand_rate'].describe().round(3).to_string())

Demand rate distribution by category × unit:
                           count   mean    std    min    25%    50%    75%     max
category final_unit                                                               
fruit    BULK_OR_UNKNOWN  4113.0  3.026  1.166  0.647  2.218  2.840  3.596  13.080
         LB               4043.0  3.584  1.429  0.757  2.613  3.313  4.259  16.323
         PINT                6.0  2.874  0.787  1.947  2.248  2.990  3.245   3.987
         UNIT              792.0  2.912  0.997  0.400  2.326  2.800  3.401  14.223
herbs    LB                 10.0  2.662  0.725  1.923  2.223  2.423  2.906   4.477
         UNIT                1.0  1.820    NaN  1.820  1.820  1.820  1.820   1.820
leafy    BULK_OR_UNKNOWN  1007.0  1.929  0.671  0.587  1.427  1.820  2.283   5.963
         LB               3699.0  3.157  1.129  0.790  2.381  2.988  3.662  20.017
         UNIT             3661.0  1.832  0.749  0.563  1.320  1.698  2.170   8.887
root     BULK_OR_UNKNOWN    10.0  1.812  0

In [10]:
# # ─────────────────────────────────────────────────────────────────
# # STEP 9 — Product fixed-effects elasticity
# # FIX 3: Product-level within-estimator removes endogeneity.
# # FIX 4: Non-promoted filter + week traffic control.
# # FIX 5: Literature override for broken positive-beta estimates.
# # ─────────────────────────────────────────────────────────────────
# AGENTS = ['leafy', 'root', 'fruit', 'herbs']

# # Literature fallback (Andreyeva et al. 2010, USDA ERS)
# LITERATURE_BETAS = {
#     'leafy': -1.76,   # highly elastic — perishable, substitutable
#     'root':  -0.69,   # moderate — staple, less substitutable
#     'fruit': -1.32,   # moderate — seasonal substitution common
#     'herbs': -0.49,   # inelastic — essential flavouring, small quantity
# }

# # ── Aggregate to product × week level ────────────────────────────
# product_weekly = (
#     trans_produce
#     .groupby(['PRODUCT_ID','WEEK_NO','category','final_unit'])
#     .agg(
#         revenue   = ('SALES_VALUE',    'sum'),
#         n_baskets = ('BASKET_ID',     'nunique'),
#         avg_price = ('standard_price', 'mean'),
#         promoted  = ('promoted',       'max'),
#     )
#     .reset_index()
# )
# product_weekly['demand_rate'] = (
#     product_weekly['revenue'] / product_weekly['n_baskets'].clip(lower=1)
# )
# product_weekly = product_weekly[product_weekly['n_baskets'] >= 3].copy()

# # ── Within-product demeaning (product fixed effects) ─────────────
# product_weekly['mean_price_prod']  = product_weekly.groupby('PRODUCT_ID')['avg_price'].transform('mean')
# product_weekly['mean_demand_prod'] = product_weekly.groupby('PRODUCT_ID')['demand_rate'].transform('mean')
# product_weekly['price_dev']  = product_weekly['avg_price']   - product_weekly['mean_price_prod']
# product_weekly['demand_dev'] = product_weekly['demand_rate'] - product_weekly['mean_demand_prod']
# product_weekly['log_price_dev']  = product_weekly['price_dev']  / product_weekly['mean_price_prod'].clip(0.01)
# product_weekly['log_demand_dev'] = product_weekly['demand_dev'] / product_weekly['mean_demand_prod'].clip(0.01)

# # ── Week-level traffic control (absorbs holiday spikes) ──────────
# week_traffic = (
#     trans_produce.groupby('WEEK_NO')['SALES_VALUE'].sum()
#     .rename('week_total').reset_index()
# )
# product_weekly = product_weekly.merge(week_traffic, on='WEEK_NO', how='left')
# product_weekly['log_traffic'] = np.log(
#     product_weekly['week_total'] / product_weekly['week_total'].mean() + 1e-9
# )

# # ── Fit elasticity per sub-category ──────────────────────────────
# DEMAND_PARAMS = {}

# for cat in AGENTS:
#     units = product_weekly[product_weekly['category'] == cat]['final_unit'].unique()
#     for unit in units:
#         key = f"{cat}_{unit}"

#         # Non-promoted rows only (removes promotional endogeneity)
#         sub = product_weekly[
#             (product_weekly['category'] == cat) &
#             (product_weekly['final_unit'] == unit) &
#             (product_weekly['promoted'] == 0) &
#             (product_weekly['n_baskets'] >= 3)
#         ].dropna(subset=['log_price_dev','log_demand_dev'])

#         base_demand = float(product_weekly[
#             (product_weekly['category'] == cat) &
#             (product_weekly['final_unit'] == unit)
#         ]['demand_rate'].mean())
#         ref_price = float(product_weekly[
#             (product_weekly['category'] == cat) &
#             (product_weekly['final_unit'] == unit)
#         ]['avg_price'].mean())

#         # Estimate promo effect separately (needs promoted rows)
#         sub_all = product_weekly[
#             (product_weekly['category'] == cat) &
#             (product_weekly['final_unit'] == unit)
#         ].dropna(subset=['log_price_dev','log_demand_dev'])
#         promo_eff = 0.10
#         if len(sub_all) > 50:
#             Xp = pd.DataFrame({'log_p': sub_all['log_price_dev'],
#                                'promo': sub_all['promoted']})
#             pm = LinearRegression().fit(Xp, sub_all['log_demand_dev'])
#             promo_eff = float(pm.coef_[1])

#         if len(sub) < 100:
#             print(f"  SKIP {key}: {len(sub)} rows → literature beta={LITERATURE_BETAS[cat]}")
#             DEMAND_PARAMS[key] = {
#                 'beta': LITERATURE_BETAS[cat], 'promo_eff': promo_eff,
#                 'base_demand': base_demand, 'ref_price': ref_price,
#                 'n_obs': len(sub), 'r2': None, 'source': 'literature',
#             }
#             continue

#         X = pd.DataFrame({'log_price_dev': sub['log_price_dev'],
#                           'log_traffic':   sub['log_traffic']})
#         m   = LinearRegression().fit(X, sub['log_demand_dev'])
#         beta = float(m.coef_[0])
#         r2   = m.score(X, sub['log_demand_dev'])

#         lit  = LITERATURE_BETAS[cat]
#         if beta >= 0 or abs(beta - lit) > 1.5:
#             print(f"  OVERRIDE {key}: β={beta:.3f} → using literature {lit:.3f}")
#             beta, source = lit, 'literature_override'
#         else:
#             source = 'ols'

#         DEMAND_PARAMS[key] = {
#             'beta': beta, 'promo_eff': promo_eff,
#             'base_demand': base_demand, 'ref_price': ref_price,
#             'n_obs': len(sub), 'r2': r2, 'source': source,
#         }
#         flag = '' if source == 'ols' else ' ⚠️'
#         print(f"{key:30s}: β={beta:6.3f} | R²={r2:.3f} | n={len(sub)}{flag}")
# ─────────────────────────────────────────────────────────────────
# STEP 9 — Category-level OLS elasticity (revised)
# Use cross-sectional variation at category × week level.
# No product fixed effects — they remove the signal we need.
# Non-promoted weeks only to reduce endogeneity.
# ─────────────────────────────────────────────────────────────────

AGENTS = ['leafy', 'root', 'fruit', 'herbs']
LITERATURE_BETAS = {'leafy': -1.76, 'root': -0.69, 'fruit': -1.32, 'herbs': -0.49}

# Aggregate to category × unit × week
# Quantity-normalized demand avoids the mechanical revenue = price * qty bias in OLS
trans_produce['qty_normalized'] = trans_produce['QUANTITY'] * trans_produce['normalized_size']

cat_weekly = (
    trans_produce
    .groupby(['WEEK_NO', 'category', 'final_unit'])
    .agg(
        revenue         = ('SALES_VALUE',      'sum'),
        qty_lbs_sum     = ('qty_normalized',   'sum'),
        n_baskets       = ('BASKET_ID',        'nunique'),
        avg_price       = ('standard_price',   'mean'),
        promoted        = ('promoted',         'mean'),
    )
    .reset_index()
)
# Use quantity-based demand for OLS (lbs or units per basket visit)
cat_weekly['demand_rate'] = cat_weekly['qty_lbs_sum'] / cat_weekly['n_baskets'].clip(1)
cat_weekly = cat_weekly[cat_weekly['n_baskets'] >= 5].copy()

# Log-normalise relative to each sub-category's own mean
cat_weekly['ref_price']  = cat_weekly.groupby(['category','final_unit'])['avg_price'].transform('mean')
cat_weekly['ref_demand'] = cat_weekly.groupby(['category','final_unit'])['demand_rate'].transform('mean')
cat_weekly['log_p'] = np.log(cat_weekly['avg_price']   / cat_weekly['ref_price']  + 1e-9)
cat_weekly['log_d'] = np.log(cat_weekly['demand_rate'] / cat_weekly['ref_demand'] + 1e-9)

DEMAND_PARAMS = {}

for cat in AGENTS:
    units = cat_weekly[cat_weekly['category'] == cat]['final_unit'].unique()
    for unit in units:
        key = f"{cat}_{unit}"

        # Non-promoted weeks only — removes main endogeneity source
        sub = cat_weekly[
            (cat_weekly['category'] == cat) &
            (cat_weekly['final_unit'] == unit) &
            (cat_weekly['promoted'] < 0.05)   # less than 5% of baskets promoted
        ].dropna(subset=['log_p','log_d'])

        base_demand = float(cat_weekly[
            (cat_weekly['category'] == cat) &
            (cat_weekly['final_unit'] == unit)
        ]['demand_rate'].mean())
        ref_price = float(cat_weekly[
            (cat_weekly['category'] == cat) &
            (cat_weekly['final_unit'] == unit)
        ]['avg_price'].mean())

        # Promo lift: compare promoted vs non-promoted weeks
        promo_weeks = cat_weekly[
            (cat_weekly['category'] == cat) &
            (cat_weekly['final_unit'] == unit) &
            (cat_weekly['promoted'] > 0.10)
        ]['demand_rate'].mean()
        non_promo_mean = base_demand
        promo_eff = float(np.log(promo_weeks / max(non_promo_mean, 1e-6) + 1e-9)) \
                    if not np.isnan(promo_weeks) else 0.10
        promo_eff = float(np.clip(promo_eff, 0.0, 0.50))  # must be positive

        if len(sub) < 20:
            print(f"  SKIP {key}: {len(sub)} non-promo weeks → literature {LITERATURE_BETAS[cat]}")
            DEMAND_PARAMS[key] = {
                'beta': LITERATURE_BETAS[cat], 'promo_eff': promo_eff,
                'base_demand': base_demand, 'ref_price': ref_price,
                'n_obs': len(sub), 'r2': None, 'source': 'literature',
            }
            continue

        # Price variation check — need at least 3% CV to have signal
        price_cv = sub['avg_price'].std() / sub['avg_price'].mean()
        if price_cv < 0.03:
            print(f"  FLAT {key}: price CV={price_cv:.3f} → literature {LITERATURE_BETAS[cat]}")
            DEMAND_PARAMS[key] = {
                'beta': LITERATURE_BETAS[cat], 'promo_eff': promo_eff,
                'base_demand': base_demand, 'ref_price': ref_price,
                'n_obs': len(sub), 'r2': None, 'source': 'literature_flat_price',
            }
            continue

        X = sub[['log_p']]
        y = sub['log_d']
        m   = LinearRegression().fit(X, y)
        beta = float(m.coef_[0])
        r2   = m.score(X, y)

        lit = LITERATURE_BETAS[cat]
        if beta >= 0 or abs(beta - lit) > 1.5:
            print(f"  OVERRIDE {key}: β={beta:.3f} → {lit:.3f} (lit)")
            beta, source = lit, 'literature_override'
        else:
            source = 'ols'

        DEMAND_PARAMS[key] = {
            'beta': beta, 'promo_eff': promo_eff,
            'base_demand': base_demand, 'ref_price': ref_price,
            'n_obs': len(sub), 'r2': r2, 'source': source,
        }
        flag = '' if source == 'ols' else ' ⚠️'
        print(f"{key:30s}: β={beta:6.3f} | R²={r2:.3f} | n={len(sub)} | CV={price_cv:.3f}{flag}")

  OVERRIDE leafy_BULK_OR_UNKNOWN: β=0.148 → -1.760 (lit)
leafy_BULK_OR_UNKNOWN         : β=-1.760 | R²=0.054 | n=73 | CV=0.111 ⚠️
  SKIP leafy_LB: 12 non-promo weeks → literature -1.76
leafy_UNIT                    : β=-2.178 | R²=0.667 | n=77 | CV=0.097
root_LB                       : β=-0.272 | R²=0.045 | n=57 | CV=0.100
  OVERRIDE root_UNIT: β=0.227 → -0.690 (lit)
root_UNIT                     : β=-0.690 | R²=0.027 | n=101 | CV=0.084 ⚠️
root_BULK_OR_UNKNOWN          : β=-0.106 | R²=0.012 | n=91 | CV=0.110
fruit_BULK_OR_UNKNOWN         : β=-0.832 | R²=0.400 | n=30 | CV=0.107
fruit_LB                      : β=-0.279 | R²=0.334 | n=38 | CV=0.173
fruit_UNIT                    : β=-0.920 | R²=0.097 | n=62 | CV=0.150
  OVERRIDE fruit_PINT: β=0.105 → -1.320 (lit)
fruit_PINT                    : β=-1.320 | R²=0.001 | n=44 | CV=0.065 ⚠️
herbs_LB                      : β=-0.715 | R²=0.101 | n=69 | CV=0.118
herbs_UNIT                    : β=-1.612 | R²=0.403 | n=93 | CV=0.099
  SKIP herbs_BULK

In [11]:
# ─────────────────────────────────────────────────────────────────
# STEP 10 — Fourier seasonality per sub-category
# FIX 6: Missing from original notebook.
# Fit on full hourly data after partialling out price effect.
# ─────────────────────────────────────────────────────────────────
for key, p in DEMAND_PARAMS.items():
    # Reconstruct cat + unit from key (handles BULK_OR_UNKNOWN underscore)
    parts = key.split('_')
    cat   = parts[0]
    unit  = '_'.join(parts[1:])

    sub = hourly[
        (hourly['category'] == cat) &
        (hourly['final_unit'] == unit)
    ].dropna(subset=['log_p_norm','log_d_norm']).copy()

    zero_season = {'sin_daily': 0.0, 'cos_daily': 0.0,
                   'sin_weekly': 0.0, 'cos_weekly': 0.0}

    if len(sub) < 50:
        DEMAND_PARAMS[key].update(zero_season)
        continue

    # Remove own-price effect, fit seasonality on residual
    sub['resid'] = sub['log_d_norm'] - p['beta'] * sub['log_p_norm']
    X_s = pd.DataFrame({
        'sin_daily':  np.sin(2 * np.pi * sub['hour_of_week'] / 24),
        'cos_daily':  np.cos(2 * np.pi * sub['hour_of_week'] / 24),
        'sin_weekly': np.sin(2 * np.pi * sub['hour_of_week'] / 168),
        'cos_weekly': np.cos(2 * np.pi * sub['hour_of_week'] / 168),
    })
    sm = LinearRegression().fit(X_s, sub['resid'])
    DEMAND_PARAMS[key].update({
        'sin_daily':  float(sm.coef_[0]),
        'cos_daily':  float(sm.coef_[1]),
        'sin_weekly': float(sm.coef_[2]),
        'cos_weekly': float(sm.coef_[3]),
    })

print("Fourier seasonality fitted.")
print("\nSample — leafy_LB seasonality coefficients:")
if 'leafy_LB' in DEMAND_PARAMS:
    p = DEMAND_PARAMS['leafy_LB']
    print(f"  sin_daily={p['sin_daily']:.4f}  cos_daily={p['cos_daily']:.4f}")
    print(f"  sin_weekly={p['sin_weekly']:.4f}  cos_weekly={p['cos_weekly']:.4f}")

Fourier seasonality fitted.

Sample — leafy_LB seasonality coefficients:
  sin_daily=0.0079  cos_daily=0.0118
  sin_weekly=-0.0582  cos_weekly=0.0101


In [12]:
# ─────────────────────────────────────────────────────────────────
# STEP 11 — Cross-price elasticity matrix (category-level version)
# Uses cat_weekly log-price pivot to estimate cross-category effects.
# ─────────────────────────────────────────────────────────────────

# Weekly category log-price pivot (non-promoted weeks only)
price_pivot = (
    cat_weekly[cat_weekly['promoted'] < 0.05]
    .pivot_table(index='WEEK_NO', columns='category', values='log_p', aggfunc='mean')
    .fillna(0)
)

E_matrix = np.zeros((4, 4))

for i, target_cat in enumerate(AGENTS):
    # Use LB unit if available (most data), otherwise first available
    avail_units = cat_weekly[cat_weekly['category'] == target_cat]['final_unit'].unique()
    primary_unit = 'LB' if 'LB' in avail_units else avail_units[0]

    sub = cat_weekly[
        (cat_weekly['category'] == target_cat) &
        (cat_weekly['final_unit'] == primary_unit) &
        (cat_weekly['promoted'] < 0.05) &
        (cat_weekly['n_baskets'] >= 5)
    ].copy()

    merged = sub.join(price_pivot, on='WEEK_NO', how='inner').dropna()
    if len(merged) < 20 or merged['log_d'].std() < 1e-6:
        print(f'  SKIP {target_cat}: insufficient data for cross-elasticity')
        continue

    other_cats = [c for c in AGENTS if c != target_cat]
    available_cross = [c for c in other_cats if c in merged.columns]
    if not available_cross:
        print(f'  SKIP {target_cat}: no cross-category price columns')
        continue

    X   = merged[available_cross]
    y   = merged['log_d']
    mod = LinearRegression().fit(X, y)

    for k, oc in enumerate(available_cross):
        j = AGENTS.index(oc)
        E_matrix[i, j] = float(mod.coef_[k])

# Cap extreme values
E_matrix = np.clip(E_matrix, -2.0, 2.0)

print('Cross-price elasticity matrix (capped at ±2.0):')
print('Row=demand_of  Col=when_price_of_col_rises_1%')
hdr = f'{"":>10}' + ''.join([f'{c:>10}' for c in AGENTS])
print(hdr)
for i, cat in enumerate(AGENTS):
    row = ''.join([f'{E_matrix[i,j]:10.3f}' for j in range(4)])
    print(f'{cat:>10}{row}')
print('Domain randomization will vary E values ±30% during training.')

  SKIP leafy: insufficient data for cross-elasticity
Cross-price elasticity matrix (capped at ±2.0):
Row=demand_of  Col=when_price_of_col_rises_1%
               leafy      root     fruit     herbs
     leafy     0.000     0.000     0.000     0.000
      root     0.496     0.000    -0.097     0.141
     fruit    -0.314    -0.179     0.000    -0.118
     herbs    -0.274     1.302     0.397     0.000
Domain randomization will vary E values ±30% during training.


In [13]:
# ─────────────────────────────────────────────────────────────────
# STEP 12 — Consolidate sub-categories → 4-category FINAL_PARAMS
# FIX 9: Missing from original. Pricing module uses 4 categories,
# not 8 sub-categories. Weighted average by observation count.
# ─────────────────────────────────────────────────────────────────
FINAL_PARAMS = {}

for cat in AGENTS:
    sub_keys = [k for k in DEMAND_PARAMS if k.startswith(cat + '_')]
    if not sub_keys:
        print(f"WARNING: no sub-categories found for {cat}")
        continue

    n_obs   = np.array([max(DEMAND_PARAMS[k]['n_obs'], 1) for k in sub_keys], float)
    weights = n_obs / n_obs.sum()

    def wavg(field, default=0.0):
        vals = [DEMAND_PARAMS[k].get(field, default) for k in sub_keys]
        return float(np.dot(vals, weights))

    FINAL_PARAMS[cat] = {
        'beta':        wavg('beta'),
        'promo_eff':   wavg('promo_eff'),
        'base_demand': wavg('base_demand'),
        'ref_price':   wavg('ref_price'),
        'sin_daily':   wavg('sin_daily'),
        'cos_daily':   wavg('cos_daily'),
        'sin_weekly':  wavg('sin_weekly'),
        'cos_weekly':  wavg('cos_weekly'),
    }

# Issue 5: Vietnamese market priors
VN_BETA_SCALE   = {'leafy':1.25, 'root':1.20, 'fruit':1.30, 'herbs':1.10}
VN_DEMAND_SCALE = {'leafy':1.40, 'root':0.95, 'fruit':0.85, 'herbs':2.10}
for cat in AGENTS:
    FINAL_PARAMS[cat]['beta']        *= VN_BETA_SCALE[cat]
    FINAL_PARAMS[cat]['base_demand'] *= VN_DEMAND_SCALE[cat]
print('Applied VN market priors.')

print("FINAL_PARAMS (4-category, demand-weighted average of sub-categories):")
print(f"{'cat':>8} {'beta':>8} {'base_demand':>12} {'ref_price':>10} {'promo_eff':>10}")
for cat, p in FINAL_PARAMS.items():
    print(f"{cat:>8} {p['beta']:>8.3f} {p['base_demand']:>12.4f} "
          f"{p['ref_price']:>10.3f} {p['promo_eff']:>10.4f}")

Applied VN market priors.
FINAL_PARAMS (4-category, demand-weighted average of sub-categories):
     cat     beta  base_demand  ref_price  promo_eff
   leafy   -2.449      52.2397      1.480     0.1386
    root   -0.457      39.4139      1.060     0.0634
   fruit   -1.126      14.3511      2.050     0.0037
   herbs   -1.348      32.0254      4.544     0.1150


In [14]:
# ─────────────────────────────────────────────────────────────────
# STEP 13 — CrossDemandModel (with all bugs fixed)
# FIX 7: Cross-price bug was: 1.0 ** self.E[i,j]  → always = 1.0
#         Fixed to: np.exp(E[i,j] * log(price_ratio))
# ─────────────────────────────────────────────────────────────────
class CrossDemandModel:
    """
    Demand model for the QMIX simulation environment.
    Inputs:  current_prices     — dict {cat: float}
             current_freshness  — dict {cat: float} ∈ [0,1]
             t                  — int, hour within episode (0–167)
             cat                — str, which category to sample
    Output:  int, units sold (Poisson sample)
    """
    AGENTS = ['leafy', 'root', 'fruit', 'herbs']

    def __init__(self, params: dict, E_matrix: np.ndarray):
        self.params = params     # FINAL_PARAMS — 4 categories
        self.E      = E_matrix   # (4, 4) cross-price elasticities

    def sample(self, current_prices: dict, current_freshness: dict,
               t: int, cat: str, promoted: bool = False) -> int:
        p = self.params[cat]
        i = self.AGENTS.index(cat)

        # 1. Own-price elasticity (log-log, relative to ref_price)
        log_p = np.log(current_prices[cat] / p['ref_price'] + 1e-9)
        own_effect = np.exp(p['beta'] * log_p)

        # 2. Promotion lift
        promo_mult = np.exp(p['promo_eff']) if promoted else 1.0

        # 3. Fourier seasonality
        season = 1.0 + (
            p['sin_daily']  * np.sin(2 * np.pi * t / 24) +
            p['cos_daily']  * np.cos(2 * np.pi * t / 24) +
            p['sin_weekly'] * np.sin(2 * np.pi * t / 168) +
            p['cos_weekly'] * np.cos(2 * np.pi * t / 168)
        )

        # 4. Freshness multiplier
        fresh_mult = 0.4 + 0.6 * float(current_freshness[cat])

        # 5. Base lambda
        lam = p['base_demand'] * own_effect * promo_mult * season * fresh_mult

        # 6. Cross-price effects — FIXED
        #    Was: cross_mult = 1.0 ** self.E[i,j]  ← always 1.0, silently broken
        #    Now: uses actual price ratio vs ref_price
        for j, other_cat in enumerate(self.AGENTS):
            if j != i and other_cat in self.params:
                log_other = np.log(
                    current_prices.get(other_cat, self.params[other_cat]['ref_price'])
                    / self.params[other_cat]['ref_price'] + 1e-9
                )
                lam *= np.exp(self.E[i, j] * log_other)

        return int(np.random.poisson(max(lam, 0.0)))

    def sample_all(self, current_prices: dict, current_freshness: dict,
                   t: int, promoted: dict = None) -> dict:
        """Sample all 4 categories in one call. Used by MarketEnv.step()."""
        promoted = promoted or {}
        return {
            cat: self.sample(current_prices, current_freshness, t, cat,
                             promoted.get(cat, False))
            for cat in self.AGENTS
        }

# Quick instantiation test
model = CrossDemandModel(FINAL_PARAMS, E_matrix)
print("CrossDemandModel instantiated successfully.")

CrossDemandModel instantiated successfully.


In [15]:
# ─────────────────────────────────────────────────────────────────
# STEP 14 — Save all outputs
# FIX 10: Missing from original notebook.
# These two files are the ONLY outputs CrossDemandModel needs.
# ─────────────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)

with open('data/demand_params.json', 'w') as f:
    json.dump(FINAL_PARAMS, f, indent=2, default=float)

np.save('data/cross_elasticity_matrix.npy', E_matrix)

with open('data/demand_params_subcategory.json', 'w') as f:
    json.dump(DEMAND_PARAMS, f, indent=2, default=float)

print("Saved:")
print("  data/demand_params.json              ← primary CrossDemandModel input")
print("  data/cross_elasticity_matrix.npy     ← 4×4 E matrix")
print("  data/demand_params_subcategory.json  ← sub-category detail for reference")

Saved:
  data/demand_params.json              ← primary CrossDemandModel input
  data/cross_elasticity_matrix.npy     ← 4×4 E matrix
  data/demand_params_subcategory.json  ← sub-category detail for reference


In [16]:
# ─────────────────────────────────────────────────────────────────
# STEP 15 — Validation tests
# All three tests must pass before using this data for training.
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("VALIDATION SUITE")
print("=" * 60)

test_prices = {c: FINAL_PARAMS[c]['ref_price'] for c in AGENTS}
test_fresh  = {'leafy': 0.85, 'root': 0.70, 'fruit': 0.90, 'herbs': 0.60}

# Test 1: All betas must be negative
print("\nTest 1: Beta signs")
all_negative = True
for cat, p in FINAL_PARAMS.items():
    sign = '✓' if p['beta'] < 0 else '✗ FAIL'
    print(f"  {cat:8s}: β={p['beta']:6.3f} {sign}")
    if p['beta'] >= 0:
        all_negative = False
print(f"  Result: {'✓ PASS' if all_negative else '✗ FAIL — check elasticity fitting'}")

# Test 2: Discount must increase demand
print("\nTest 2: Discount increases demand (leafy price -30%)")
disc_prices = test_prices.copy()
disc_prices['leafy'] = test_prices['leafy'] * 0.70
np.random.seed(42)
base_samples = [model.sample(test_prices, test_fresh, 9, 'leafy') for _ in range(1000)]
disc_samples = [model.sample(disc_prices, test_fresh, 9, 'leafy') for _ in range(1000)]
base_mean, disc_mean = np.mean(base_samples), np.mean(disc_samples)
lift = (disc_mean / max(base_mean, 1e-6) - 1) * 100
ok   = disc_mean > base_mean
print(f"  Baseline: {base_mean:.3f}  Discounted: {disc_mean:.3f}  Lift: {lift:.1f}%")
print(f"  Result: {'✓ PASS' if ok else '✗ FAIL — demand not responding to price'}")

# Test 3: Low freshness reduces demand
print("\nTest 3: Low freshness reduces demand")
fresh_good = test_fresh.copy()
fresh_poor = test_fresh.copy(); fresh_poor['fruit'] = 0.15
good_samples = [model.sample(test_prices, fresh_good, 12, 'fruit') for _ in range(1000)]
poor_samples = [model.sample(test_prices, fresh_poor, 12, 'fruit') for _ in range(1000)]
ok2 = np.mean(poor_samples) < np.mean(good_samples)
print(f"  Fresh fruit: {np.mean(good_samples):.3f}  Stale fruit: {np.mean(poor_samples):.3f}")
print(f"  Result: {'✓ PASS' if ok2 else '✗ FAIL — freshness not reducing demand'}")

# Test 4: Cross-price effects are active (not zero)
print("\nTest 4: Cross-price effects are non-zero")
high_leafy = test_prices.copy(); high_leafy['leafy'] = test_prices['leafy'] * 1.30
baseline_fruit = np.mean([model.sample(test_prices, test_fresh, 12, 'fruit') for _ in range(1000)])
crossfx_fruit  = np.mean([model.sample(high_leafy,  test_fresh, 12, 'fruit') for _ in range(1000)])
cross_active = abs(crossfx_fruit - baseline_fruit) > 0.001
print(f"  Fruit demand @ baseline leafy: {baseline_fruit:.3f}")
print(f"  Fruit demand @ +30% leafy:     {crossfx_fruit:.3f}")
print(f"  Result: {'✓ PASS — cross-price active' if cross_active else '✗ FAIL — cross-price still broken'}")

print("\n" + "=" * 60)
all_pass = all_negative and ok and ok2 and cross_active
print(f"OVERALL: {'✓ ALL TESTS PASSED — ready for QMIX training' if all_pass else '✗ SOME TESTS FAILED — fix before training'}")
print("=" * 60)

VALIDATION SUITE

Test 1: Beta signs
  leafy   : β=-2.449 ✓
  root    : β=-0.457 ✓
  fruit   : β=-1.126 ✓
  herbs   : β=-1.348 ✓
  Result: ✓ PASS

Test 2: Discount increases demand (leafy price -30%)
  Baseline: 47.920  Discounted: 115.368  Lift: 140.8%
  Result: ✓ PASS

Test 3: Low freshness reduces demand
  Fresh fruit: 13.282  Stale fruit: 6.793
  Result: ✓ PASS

Test 4: Cross-price effects are non-zero


  Fruit demand @ baseline leafy: 13.271
  Fruit demand @ +30% leafy:     12.249
  Result: ✓ PASS — cross-price active

OVERALL: ✓ ALL TESTS PASSED — ready for QMIX training


In [17]:
# ─────────────────────────────────────────────────────────────────
# STEP 16 — Loader for MarketEnv
# Copy this function into env/demand.py
# ─────────────────────────────────────────────────────────────────
def load_demand_model(
    params_path='data/demand_params.json',
    matrix_path='data/cross_elasticity_matrix.npy'
) -> CrossDemandModel:
    with open(params_path) as f:
        params = json.load(f)
    E = np.load(matrix_path)
    return CrossDemandModel(params, E)

# Test the loader
dm = load_demand_model()
sample_all = dm.sample_all(
    {c: dm.params[c]['ref_price'] for c in dm.AGENTS},
    {'leafy': 0.8, 'root': 0.7, 'fruit': 0.9, 'herbs': 0.6},
    t=9
)
print("Sample output from load_demand_model() at t=9 (morning):")
for cat, units in sample_all.items():
    print(f"  {cat:8s}: {units} units sold")
print("\nThis is what MarketEnv.step() calls each tick.")

Sample output from load_demand_model() at t=9 (morning):
  leafy   : 43 units sold
  root    : 29 units sold
  fruit   : 12 units sold
  herbs   : 31 units sold

This is what MarketEnv.step() calls each tick.
